In [3]:
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from agent_graph import StagePlayWriter,input_message, StagePlayState
from langgraph.types import Command
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
import time

from uuid import uuid4

### Setup agent graph

In [4]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")
# tools = [get_character_description, create_character, human_assistance]

thread_id = uuid4()

graph_config: RunnableConfig = RunnableConfig({"configurable": {"thread_id": thread_id}})

playwriter = StagePlayWriter(
    llm=llm,
    themes= """Loss of innocence, Becoming Psychologically whole, Jungian Psychology, Bildung""",
    vibe= """Subtly, Weird and funky""",
    setting= "Tam Tamoree, fictional town in German Bavaria",
    number_of_chapters= 4
)
conn_checkptr = "db/graph_checkpoints/checkpoints.db"


### Graph invoke and continue functions

In [5]:
async def start_graph(init_message: str ,graph_config:  RunnableConfig) -> dict[str, str]:
    """Start the agent application
    """
    async with AsyncSqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
        async for event in (playwriter
                            .build_graph(checkpointer)
                            .astream(input=input_message(init_message), config= graph_config, stream_mode="values")):

            context = event["context"][-1]
            if isinstance(context, tuple):
                print(event)
            else:
                context.pretty_print()

    return {"status": "Graph started, may be paused"}


async def resume_graph(input: str, graph_config) -> dict[str, str]:
    """Resume graph after break from human in the loop tool call
    """
    resume_input = Command(resume= {"data": input}) 
    async with AsyncSqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
        async for event in (playwriter
                            .build_graph(checkpointer)
                            .astream(input=resume_input, config= graph_config, stream_mode="values")):

            context = event["context"][-1]
            if isinstance(context, tuple):
                print(event)
            else:
                context.pretty_print()
    return {"status": "Resumed, may stopp again"}



### Call graph 

In [6]:
await start_graph(init_message= """Narrator: It is a sunny wistful day in Tam Tamouree.
        Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
        Luna:
        """, graph_config= graph_config)

================================ Human Message =================================

Narrator: It is a sunny wistful day in Tam Tamouree.
        Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
        Luna:
        
================================== Ai Message ==================================

Please provide me with the current state of the play, including any preceding context and the new line you would like me to write for the given character or narrator.
================================== Ai Message ==================================

### Current State of the Play

- **Narrator**:  
   It is a misty evening in Tam Tamoree, where shadows dance along the cobblestone streets. The faint sound of laughter echoes from a nearby tavern as the townspeople gather to escape the chill. 

- **Luna**:  
   (with a hint of sass and sarcasm) "Looks like another glorious evening in Tam Tamoree! Where shall we waste our youth and good looks 

{'status': 'Graph started, may be paused'}

In [7]:
await resume_graph("""Luna:
                   Oh my god! My blood is ants. I am dying Swedenborg. Oh! The pain!! I can't see. The creepy crawlies, they're within me. Help! """, 
                   graph_config= graph_config)

================================ Human Message =================================

Narrator: 


{'status': 'Resumed, may stopp again'}

In [8]:
await resume_graph("""Swedenborg:
                   Whatever! You're being weird today """, graph_config)

================================ Human Message =================================

Narrator: 


{'status': 'Resumed, may stopp again'}

In [9]:
await resume_graph("""Swedenborg:
                   Whatever! You're being weird today """, graph_config)

================================ Human Message =================================

Narrator: 


{'status': 'Resumed, may stopp again'}

In [10]:
async with AsyncSqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
    playwriter.build_graph()

TypeError: StagePlayWriter.build_graph() missing 1 required positional argument: 'checkpointer'